In [ ]:
%pip install weaviate-client

In [2]:
import weaviate
import json
from weaviate.classes.config import Configure, DataType, Property, VectorDistances
from weaviate.classes.query import Filter, MetadataQuery
from dotenv import load_dotenv
import os


d:\epam\sadhaka\module_3\rag\vectordb\.venv\Lib\site-packages\authlib\integrations\httpx_client\assertion_client.py:5: AuthlibDeprecationWarning: The httpx module is deprecated; please use httpx2 instead.
  from ._compat import httpx2


In [3]:
load_dotenv()

True

In [4]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.getenv("WEAVIATE_REST_ENDPOINT"),
    auth_credentials=os.getenv("WEAVIATE_API_KEY_ADMIN")
)

print("Weaviate ready:", client.is_ready())

Weaviate ready: True


In [7]:

collection = client.collections.create(
    name="MyCollection",
    properties=[
        Property(name="title", data_type=DataType.TEXT),# will use both the properties to generate vectors
        Property(name="foo", data_type=DataType.INT)
    ],
    vector_config=Configure.Vectors.text2vec_weaviate(
        vector_index_config=Configure.VectorIndex.hfresh(
            distance_metric=VectorDistances.COSINE
        )
    ),
)

print("created collection", collection.name)

created collection MyCollection


In [8]:
data = [
    {
        "title": "This is ram's boat.",
        "foo": 99,
    },
    {
        "title": "Boat is India's one of the famous headphone brand.",
        "foo": 77
    },
    {
        "title": "Ram is a dump boy. He can't even read a single sentence properly",
        "foo": 55
    },
    {
        "title": "Patrick is my dog name.",
        "foo": 33
    },
    {
        "title": "Dogs are the most loyal animal to humans.",
        "foo": 11        
    },
]

object_uuids = []
for item in data:
    uuid = collection.data.insert(
        properties={
            "title": item["title"],
            "foo": item["foo"],
        }
    )
    object_uuids.append(uuid)

print(f"Inserted {len(object_uuids)} objects.")


Inserted 5 objects.


In [9]:
print(object_uuids)

[UUID('09646e7c-a3fd-45f5-ad94-bfca8a02ec2b'), UUID('47438f09-f330-4a57-a86a-f9f23708b8f4'), UUID('b94aef5b-3c61-4269-ba4c-d4feb449f439'), UUID('dc3cefdc-9f49-4093-a0e2-19a3c9287adf'), UUID('f4c78b47-7550-4d09-ae12-8949fc18b571')]


# Querying through text

In [ ]:
#vector similarity search
response = collection.query.near_text(
    query="I need to cross this river.", # search query
    limit=2, # limit
    offset=1, # itne results chood ke dikhao
    include_vector=True, # to get the vector
    return_metadata=MetadataQuery(distance=True, score=True), # to include the distance from query
    return_properties=["title"], # specify the properties that you want to get else give all the properties
    filters=Filter.by_property("foo").greater_than(44) #you can apply filter to your query result base on some other property, meta data, time of creation and etc
)

for o in response.objects:
    print(o.properties)
    print(o.metadata.distance)
    print(o.vector["default"][0:5])
    print(o.metadata.score)

{'title': "Ram is a dump boy. He can't even read a single sentence properly"}
0.8667535781860352
[0.04543877765536308, 0.006820629816502333, 0.10435515642166138, -0.029073115438222885, -0.05953250080347061]
0.0
{'title': "Boat is India's one of the famous headphone brand."}
0.9195846915245056
[-0.008541516028344631, -0.06512215733528137, -0.014314436353743076, 0.05926397442817688, -0.008060021325945854]
0.0


In [30]:
vector = response.objects[0].vector["default"]
uuid = response.objects[0].uuid

print(vector,uuid, sep='\n')

[0.04543877765536308, 0.006820629816502333, 0.10435515642166138, -0.029073115438222885, -0.05953250080347061, -0.05121489241719246, 0.050098177045583725, 0.03850743919610977, 0.009342866949737072, -0.041934601962566376, 0.02533789351582527, 0.036023709923028946, 0.015229691751301289, -0.029381174594163895, -0.042512211948633194, -0.0027605020441114902, -0.0887211412191391, -0.008389808237552643, -0.06946741789579391, -0.03966266289353371, -0.05090683326125145, 0.009597978554666042, 0.025357147678732872, 0.005063727963715792, -0.034945499151945114, 0.0823289006948471, -0.0033645874354988337, 0.00307578151114285, -0.013901185244321823, 0.053717877715826035, -0.035985201597213745, 0.038622960448265076, 0.033308934420347214, -0.02335476130247116, 0.004883224610239267, -0.06011011078953743, 0.04805728420615196, -0.02302744798362255, -0.03059415891766548, 0.025491924956440926, -0.011080515570938587, 0.013265812769532204, 0.03812236338853836, -0.07270204275846481, 0.0029578525573015213, -0.00

# Querying through Vector

In [32]:
response = collection.query.near_vector(
    near_vector=vector,
    limit=1,
    return_properties=["title"],
    filters=Filter.by_property("foo").greater_or_equal(44)
)

for o in response.objects:
    print(o.properties["title"])

Ram is a dump boy. He can't even read a single sentence properly


# Querying through UUID

In [36]:
response = collection.query.near_object(
    near_object=uuid,
    limit=2,
    return_properties=['title']
)

print(response.objects[0].properties['title'])

Ram is a dump boy. He can't even read a single sentence properly


In [39]:

import json

def json_print(data):
    print(json.dumps(data, indent=4, ensure_ascii=False))


In [40]:
json_print(client.get_meta())

{
    "grpcMaxMessageSize": 104858000,
    "hostname": "http://[::]:8080",
    "license": {
        "documentationHref": "https://docs.weaviate.io/deploy/enterprise",
        "edition": "enterprise",
        "licenseId": "lic_06GH1AHSN9X8FG04Y3N3AG5P2W",
        "status": "valid"
    },
    "modules": {
        "backup-s3": {
            "bucketName": "prod-wcs-data-plane-eu-central-1-workloads-backups",
            "endpoint": "s3.amazonaws.com",
            "rootName": "3c63b57c-c797-4995-8a6f-c13e16463d64",
            "useSSL": true
        },
        "generative-anthropic": {
            "documentationHref": "https://docs.anthropic.com/en/api/getting-started",
            "name": "Generative Search - Anthropic"
        },
        "generative-anyscale": {
            "documentationHref": "https://docs.anyscale.com/endpoints/overview",
            "name": "Generative Search - Anyscale"
        },
        "generative-aws": {
            "documentationHref": "https://docs.aws.amazon.c